In [17]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# =====================================================================
# SECCIÓN 1: DIFERENCIACIÓN E INTERPOLACIÓN NUMÉRICA
# =====================================================================

def primera_derivada_3_puntos(f0, f1, f2, h, s):
    """
    Fórmula general de la primera derivada de tres puntos (n=2).
    Calcula la derivada aproximada usando tres puntos discretos.
    
    Parámetros:
    f0, f1, f2 : Valores de la función (o vectores de estado) en puntos consecutivos.
    h : Tamaño del paso de integración.
    s : Parámetro de posición para la derivada.
    """
    termino_1 = (-3 + 2*s) * f0
    termino_2 = 4 * (1 - s) * f1
    termino_3 = (-1 + 2*s) * f2
    
    return (termino_1 + termino_2 + termino_3) / (2 * h)

def segunda_derivada_3_puntos(f0, f1, f2, h):
    """
    Fórmula de diferencia para la segunda derivada (n=2).
    
    Parámetros:
    f0, f1, f2 : Valores de la función (o vectores de estado) en puntos consecutivos.
    h : Tamaño del paso temporal.
    """
    return (f0 - 2*f1 + f2) / (h**2)

def interpolador_lagrange(t_eval, t_points, state_points):
    """
    Interpola datos de estado entre pasos de tiempo discretos utilizando 
    la metodología de interpolación de Lagrange.
    Útil cuando el paso de integración h es demasiado grueso para capturar 
    dinámicas de encuentros cercanos.
    
    Parámetros:
    t_eval : Tiempo(s) exacto(s) donde se desea evaluar el estado (escalar o array).
    t_points : Array con los tiempos discretos conocidos (nodos).
    state_points : Array con los vectores de estado correspondientes a t_points.
    """
    t_eval = np.atleast_1d(t_eval)
    n = len(t_points)
    
    # Preparar el array de salida con la forma adecuada
    interpolated_states = np.zeros((len(t_eval),) + np.shape(state_points[0]))
    
    for i, t in enumerate(t_eval):
        estado_t = np.zeros_like(state_points[0], dtype=float)
        for j in range(n):
            # Calcular el polinomio base de Lagrange l_j(x)
            l_j = 1.0
            for m in range(n):
                if m != j:
                    l_j *= (t - t_points[m]) / (t_points[j] - t_points[m])
            estado_t += l_j * state_points[j]
        interpolated_states[i] = estado_t
        
    return interpolated_states if len(t_eval) > 1 else interpolated_states[0]

# =====================================================================
# SECCIÓN 2: MOTOR DE FUERZAS Y POTENCIAL GRAVITACIONAL
# =====================================================================

def calcular_aceleraciones(masas, posiciones, G):
    """
    Calcula la aceleración gravitacional absoluta en 3D para un sistema de N cuerpos.
    
    Esta función encapsula la lógica del potencial gravitacional de forma 
    modular e independiente del integrador, cumpliendo con los requisitos 
    de diseño del motor de simulación.
    
    Parámetros:
    masas      : Array 1D de tamaño N con las masas de los cuerpos.
    posiciones : Array 2D de tamaño (N, 3) con las posiciones cartesianas (x, y, z).
    G          : Constante gravitacional (debe coincidir con el sistema de unidades).
    
    Retorna:
    aceleraciones : Array 2D de tamaño (N, 3) con los vectores de aceleración.
    """
    N = len(masas)
    aceleraciones = np.zeros((N, 3))
    
    for i in range(N):
        # Vector de distancias relativas desde todos los cuerpos j hacia el cuerpo i (r_AB = x_A - x_B)
        r_vectores = posiciones[i] - posiciones
        
        # Norma euclidiana (magnitud) de las distancias
        distancias = np.linalg.norm(r_vectores, axis=1)
        
        # Evitar la singularidad y división por cero cuando i == j
        distancias[i] = np.inf
        
        # Cálculo del factor escalar: G * m_B / r_AB^3
        # Se aprovecha la vectorización de numpy para operar sobre todos los cuerpos simultáneamente
        factor = G * masas / (distancias**3)
        
        # Suma vectorial de los gradientes gravitacionales producidos por todos los otros cuerpos
        aceleraciones[i] = -np.sum(factor[:, np.newaxis] * r_vectores, axis=0)
        
    return aceleraciones

def calcular_energia_potencial(masas, posiciones, G):
    """
    Calcula la energía potencial gravitacional total del sistema N-cuerpos.
    Necesario para la validación física y el análisis de conservación de energía.
    """
    N = len(masas)
    U = 0.0
    for i in range(N):
        for j in range(i + 1, N):
            distancia = np.linalg.norm(posiciones[i] - posiciones[j])
            # Interacción de a pares (pairwise)
            U -= G * masas[i] * masas[j] / distancia
    return U

# =====================================================================
# SECCIÓN 3: MÓDULO DE INTEGRADORES NUMÉRICOS (SOLVENTES ODE)
# =====================================================================


def paso_rk4(masas, posiciones, velocidades, h, G):
    """
    Ejecuta un paso de integración utilizando el método de Runge-Kutta de 4.º Orden (RK4).
    Equilibra la sobrecarga computacional con una alta precisión local.
    
    Parámetros:
    masas       : Array 1D con las masas de los N cuerpos.
    posiciones  : Array 2D (N, 3) con las posiciones actuales.
    velocidades : Array 2D (N, 3) con las velocidades actuales.
    h           : Tamaño del paso temporal.
    G           : Constante gravitacional.
    
    Retorna:
    (posiciones_nuevas, velocidades_nuevas)
    """
    # k1
    v_k1 = velocidades
    a_k1 = calcular_aceleraciones(masas, posiciones, G)
    
    # k2
    pos_k2 = posiciones + 0.5 * h * v_k1
    v_k2 = velocidades + 0.5 * h * a_k1
    a_k2 = calcular_aceleraciones(masas, pos_k2, G)
    
    # k3
    pos_k3 = posiciones + 0.5 * h * v_k2
    v_k3 = velocidades + 0.5 * h * a_k2
    a_k3 = calcular_aceleraciones(masas, pos_k3, G)
    
    # k4
    pos_k4 = posiciones + h * v_k3
    v_k4 = velocidades + h * a_k3
    a_k4 = calcular_aceleraciones(masas, pos_k4, G)
    
    # Actualización final (vectorizada en 3D para todos los cuerpos)
    posiciones_nuevas = posiciones + (h / 6.0) * (v_k1 + 2*v_k2 + 2*v_k3 + v_k4)
    velocidades_nuevas = velocidades + (h / 6.0) * (a_k1 + 2*a_k2 + 2*a_k3 + a_k4)
    
    return posiciones_nuevas, velocidades_nuevas

def paso_simplectico(masas, posiciones, velocidades, aceleraciones_actuales, h, G):
    """
    Ejecuta un paso de integración utilizando un Integrador Simpléctico (Kick-Drift-Kick).
    Recomendado para la estabilidad de larga duración y la preservación del Hamiltoniano.
    
    Parámetros:
    masas                  : Array 1D con las masas.
    posiciones             : Array 2D (N, 3) con las posiciones actuales.
    velocidades            : Array 2D (N, 3) con las velocidades actuales.
    aceleraciones_actuales : Array 2D (N, 3) con las aceleraciones calculadas en el paso anterior.
    h                      : Tamaño del paso temporal.
    G                      : Constante gravitacional.
    
    Retorna:
    (posiciones_nuevas, velocidades_nuevas, aceleraciones_nuevas)
    """
    # Kick 1: Actualizar velocidades medio paso
    v_media = velocidades + 0.5 * h * aceleraciones_actuales
    
    # Drift: Actualizar posiciones un paso completo usando la velocidad media
    posiciones_nuevas = posiciones + h * v_media
    
    # Calcular las nuevas aceleraciones basadas en las nuevas posiciones
    aceleraciones_nuevas = calcular_aceleraciones(masas, posiciones_nuevas, G)
    
    # Kick 2: Actualizar velocidades el resto del medio paso
    velocidades_nuevas = v_media + 0.5 * h * aceleraciones_nuevas
    
    return posiciones_nuevas, velocidades_nuevas, aceleraciones_nuevas

# =====================================================================
# SECCIÓN 4: MÓDULO DE DETECCIÓN DE EVENTOS Y RAÍCES
# =====================================================================

def buscar_raiz_biseccion(func_objetivo, t_a, t_b, tol=1e-8, max_iter=100):
    """
    Implementa el método de bisección para identificar cruces orbitales o ápsides.
    Encuentra el valor de 't' tal que func_objetivo(t) == 0 dentro del intervalo [t_a, t_b].
    
    En el contexto del simulador, 'func_objetivo' típicamente utilizará la 
    función 'interpolador_lagrange' (de la Sección 1) para encontrar el instante 
    exacto en que un cuerpo cruza una coordenada (ej. Z = 0) o alcanza 
    una velocidad radial nula (ápsides).
    
    Parámetros:
    func_objetivo : Función llamable (callable) que evalúa la condición en el tiempo t.
    t_a, t_b      : Límites del intervalo de tiempo donde se detectó el evento (cambio de signo).
    tol           : Tolerancia o error máximo permitido para la raíz.
    max_iter      : Número máximo de divisiones del intervalo.
    
    Retorna:
    El valor de t donde ocurre el evento (la raíz).
    """
    f_a = func_objetivo(t_a)
    f_b = func_objetivo(t_b)
    
    # Verificación estricta del teorema de Bolzano
    if f_a * f_b > 0:
        raise ValueError("El intervalo [t_a, t_b] no contiene un cambio de signo garantizado.")
        
    t_izq = t_a
    t_der = t_b
    
    for _ in range(max_iter):
        t_medio = 0.5 * (t_izq + t_der)
        f_medio = func_objetivo(t_medio)
        
        # Condición de salida: si estamos suficientemente cerca del cero o 
        # el tamaño del intervalo es menor que la tolerancia
        if abs(f_medio) < tol or 0.5 * (t_der - t_izq) < tol:
            return t_medio
            
        # Reasignación de límites manteniendo el cambio de signo
        if f_a * f_medio < 0:
            # La raíz está en la mitad izquierda
            t_der = t_medio
        else:
            # La raíz está en la mitad derecha
            t_izq = t_medio
            f_a = f_medio
            
    # Si se agota el número máximo de iteraciones, retornamos la mejor aproximación
    return 0.5 * (t_izq + t_der)

# =====================================================================
# SECCIÓN 5: MÓDULO DE DIAGNÓSTICO, CONSERVACIÓN Y ANÁLISIS DE ERROR
# =====================================================================

def calcular_momento_angular(masas, posiciones, velocidades):
    """
    Calcula el vector de Momento Angular Total (L) del sistema N-cuerpos.
    Representa la suma vectorial del momento angular de cada cuerpo.
    
    Parámetros:
    masas       : Array 1D con las masas.
    posiciones  : Array 2D (N, 3) con las posiciones.
    velocidades : Array 2D (N, 3) con las velocidades.
    
    Retorna:
    L_total : Array 1D (3,) con las componentes (Lx, Ly, Lz) del momento angular.
    """
    L_total = np.zeros(3)
    for i in range(len(masas)):
        # Producto cruz tridimensional: r x v
        momento_i = masas[i] * np.cross(posiciones[i], velocidades[i])
        L_total += momento_i
    return L_total

def calcular_energia_cinetica(masas, velocidades):
    """
    Calcula la Energía Cinética Total (K) del sistema.
    """
    K = 0.0
    for i in range(len(masas)):
        # Magnitud de la velocidad al cuadrado usando producto punto
        v_cuadrado = np.dot(velocidades[i], velocidades[i])
        K += 0.5 * masas[i] * v_cuadrado
    return K

def calcular_energia_total(masas, posiciones, velocidades, G):
    """
    Calcula la Energía Total (E) del sistema.
    Es la suma de la energía cinética y la energía potencial gravitacional.
    """
    K = calcular_energia_cinetica(masas, velocidades)
    U = calcular_energia_potencial(masas, posiciones, G) # Requiere la Sección 2
    return K + U

def error_truncamiento_e3(h, s, f_cuarta_derivada_xi):
    """
    Calcula el error de truncamiento teórico E'_3 basándose en la ecuación 0.7.
    Se utiliza para justificar matemáticamente la selección del paso h frente a
    la deriva real medida (solver drift).
    
    Parámetros:
    h                    : Tamaño del paso temporal.
    s                    : Parámetro posicional de la malla de interpolación.
    f_cuarta_derivada_xi : Aproximación de la 4ta derivada evaluada en xi ( f^{iv}(xi) ).
    """
    # Término polinomial: (3 - 11s + 9s^2 - 2s^3)
    polinomio = 3 - 11*s + 9*(s**2) - 2*(s**3)
    
    # Factor inicial: -h^3 / 12
    factor_h = -(h**3) / 12.0
    
    error_teorico = factor_h * polinomio * f_cuarta_derivada_xi
    return error_teorico

# =====================================================================
# SECCIÓN 6: MÓDULO DE REGISTRO (LOGGING) Y VISUALIZACIÓN
# =====================================================================

def simular_y_registrar(masas, pos_ini, vel_ini, t_final, h, G, integrador='rk4'):
    """
    Ejecuta la simulación completa guardando el historial de estados.
    Cumple con el requisito de "Timestamped Logging": registrar tiempo, 
    vectores de estado y energía total en cada paso.
    
    Parámetros:
    integrador: 'rk4' o 'simplectico'
    """
    pasos = int(np.ceil(t_final / h))
    N = len(masas)
    
    # Inicialización de arreglos de registro (Logs)
    log_t = np.zeros(pasos)
    log_pos = np.zeros((pasos, N, 3))
    log_vel = np.zeros((pasos, N, 3))
    log_energia = np.zeros(pasos)
    log_momento_L = np.zeros((pasos, 3)) # Se guarda para graficar L total
    
    # Condiciones iniciales
    pos_actual = np.copy(pos_ini)
    vel_actual = np.copy(vel_ini)
    accel_actual = calcular_aceleraciones(masas, pos_actual, G) # Para el simpléctico
    
    for i in range(pasos):
        t = i * h
        
        # 1. Registrar el estado actual
        log_t[i] = t
        log_pos[i] = pos_actual
        log_vel[i] = vel_actual
        log_energia[i] = calcular_energia_total(masas, pos_actual, vel_actual, G)
        log_momento_L[i] = calcular_momento_angular(masas, pos_actual, vel_actual)
        
        # 2. Propagar al siguiente paso temporal
        if integrador == 'rk4':
            pos_actual, vel_actual = paso_rk4(masas, pos_actual, vel_actual, h, G)
        elif integrador == 'simplectico':
            pos_actual, vel_actual, accel_actual = paso_simplectico(
                masas, pos_actual, vel_actual, accel_actual, h, G)
        else:
            raise ValueError("Integrador no reconocido.")
            
    return log_t, log_pos, log_vel, log_energia, log_momento_L


def graficar_validacion_fisica(log_t, log_energia, log_momento_L):
    """
    Genera el 'Validation Suite': Gráficos de múltiples paneles mostrando 
    la evolución de la Energía Total (E) y el Momento Angular Total (L).
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    
    # Panel 1: Energía Total
    ax1.plot(log_t, log_energia, color='red', label='Energía Total (E)')
    ax1.set_ylabel("Energía")
    ax1.set_title("Validación Física: Conservación del Sistema")
    ax1.grid(True)
    ax1.legend()
    
    # Panel 2: Momento Angular Total (Magnitud y componentes)
    magnitud_L = np.linalg.norm(log_momento_L, axis=1)
    ax2.plot(log_t, magnitud_L, color='blue', label='Magnitud ||L||')
    ax2.plot(log_t, log_momento_L[:, 0], '--', label='Lx', alpha=0.5)
    ax2.plot(log_t, log_momento_L[:, 1], '--', label='Ly', alpha=0.5)
    ax2.plot(log_t, log_momento_L[:, 2], '--', label='Lz', alpha=0.5)
    ax2.set_xlabel("Tiempo de simulación")
    ax2.set_ylabel("Momento Angular")
    ax2.grid(True)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

def graficar_trayectorias_3d(log_pos, nombres=None, titulo="Visualización de Órbitas 3D", radios_cuerpos=None, factor_escala=1.0):
    """
    Genera la visualización 3D con escalas reales (1:1:1) y dibuja los cuerpos celestes
    como esferas en su posición final, permitiendo magnificarlos si es necesario.
    """
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    pasos, N, _ = log_pos.shape
    
    # 1. Dibujar cada trayectoria y el cuerpo físico
    for i in range(N):
        x = log_pos[:, i, 0]
        y = log_pos[:, i, 1]
        z = log_pos[:, i, 2]
        
        etiqueta = nombres[i] if nombres else f"Cuerpo {i}"
        
        # Trayectoria
        ax.plot(x, y, z, label=etiqueta)
        
        # Dibujar el cuerpo como esfera (en su última posición)
        if radios_cuerpos is not None and i < len(radios_cuerpos) and radios_cuerpos[i] > 0:
            R = radios_cuerpos[i] * factor_escala
            u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
            x_esfera = R * np.cos(u) * np.sin(v) + x[-1]
            y_esfera = R * np.sin(u) * np.sin(v) + y[-1]
            z_esfera = R * np.cos(v) + z[-1]
            
            # Usar un color distinto para el Sol (índice 0 en la Tarea 3)
            color_esfera = 'orange' if (nombres and nombres[i] == "Sol") else 'dodgerblue'
            ax.plot_surface(x_esfera, y_esfera, z_esfera, color=color_esfera, alpha=0.7, edgecolor='none')
        else:
            # Si no hay radio, solo poner un punto
            ax.scatter(x[-1], y[-1], z[-1], marker='o')

    # 2. CORRECCIÓN DE ESCALA ESPACIAL (Cubo perfecto)
    max_dist = np.max(np.abs(log_pos))
    ax.set_xlim([-max_dist, max_dist])
    ax.set_ylim([-max_dist, max_dist])
    ax.set_zlim([-max_dist, max_dist])
    
    try:
        ax.set_box_aspect([1, 1, 1])
    except AttributeError:
        pass
        
    ax.set_xlabel('Posición X (m)')
    ax.set_ylabel('Posición Y (m)')
    ax.set_zlabel('Posición Z (m)')
    ax.set_title(titulo)
    ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1))
    plt.show()

def graficar_evolucion_deriva(log_t, drift_array):
    """Grafica cómo se acumula el error numérico a lo largo del tiempo (Tarea 1)."""
    plt.figure(figsize=(10, 5))
    plt.plot(log_t / 86400, drift_array / 1000, color='purple', linewidth=2)
    plt.title("Evolución del Solver Drift (RK4 vs Analítico)")
    plt.xlabel("Tiempo (Días)")
    plt.ylabel("Error de Posición (km)")
    plt.grid(True)
    plt.show()

def graficar_inclinacion_z(log_t, log_pos, indices, nombres):
    """Compara la elevación en Z para demostrar órbitas no coplanares (Tarea 3)."""
    plt.figure(figsize=(10, 5))
    for idx, nombre in zip(indices, nombres):
        # Extraer la coordenada Z (índice 2) a lo largo del tiempo
        z_coords = log_pos[:, idx, 2] / 1e9  # Convertir a millones de km
        plt.plot(log_t / 86400, z_coords, label=nombre)
        
    plt.title("Oscilación del Plano Orbital (Coordenada Z)")
    plt.xlabel("Tiempo (Días)")
    plt.ylabel("Elevación Z (10^6 km)")
    plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
    plt.legend()
    plt.grid(True)
    plt.show()

def graficar_distancia_relativa(log_t, log_pos, idx1, idx2, nombre_sis):
    """Muestra la distancia entre dos cuerpos para evaluar la estabilidad (Tarea 3)."""
    distancias = np.linalg.norm(log_pos[:, idx1, :] - log_pos[:, idx2, :], axis=1)
    
    plt.figure(figsize=(10, 5))
    plt.plot(log_t / 86400, distancias / 1000, color='green')
    plt.title(f"Estabilidad de Distancia Relativa: {nombre_sis}")
    plt.xlabel("Tiempo (Días)")
    plt.ylabel("Distancia (km)")
    plt.grid(True)
    plt.show()

# Datos base
G_Tierra = 3.986004418e14  # m^3/s^2
M_Tierra = 5.972e24        # kg
G_const = 6.67430e-11      # Constante gravitacional universal

# =====================================================================
# SECCIÓN 6.5: UTILIDADES ORBITALES Y SOLUCIONADOR DE KEPLER
# =====================================================================

def resolver_anomalia_excentrica(M, e, tol=1e-8, max_iter=100):
    """
    Implementa un método iterativo (Newton-Raphson) para resolver la 
    ecuación trascendental de Kepler: M = E - e*sin(E).
    Necesario para el Solucionador Universal de Kepler.
    """
    # Estimación inicial
    E = M if e < 0.8 else np.pi
    
    for _ in range(max_iter):
        f = E - e * np.sin(E) - M
        f_prima = 1.0 - e * np.cos(E)
        dE = f / f_prima
        E -= dE
        if abs(dE) < tol:
            break
    return E

def kepler_a_cartesiano(a, e, i_deg, Omega_deg, omega_deg, M, mu):
    """
    Convierte los 6 elementos orbitales Keplerianos (a, e, i, Omega, omega, M)
    en vectores de estado cartesianos (r, v) en el marco inercial.
    """
    # Convertir grados a radianes
    i = np.radians(i_deg)
    Omega = np.radians(Omega_deg)
    omega = np.radians(omega_deg)
    
    # 1. Resolver la ecuación de Kepler
    E = resolver_anomalia_excentrica(M, e)
    
    # 2. Posición y velocidad en el plano orbital (con la Anomalía Verdadera nu)
    nu = 2.0 * np.arctan(np.sqrt((1.0 + e) / (1.0 - e)) * np.tan(E / 2.0))
    r_mag = a * (1.0 - e * np.cos(E))
    
    r_orb = np.array([r_mag * np.cos(nu), r_mag * np.sin(nu), 0.0])
    
    p = a * (1.0 - e**2)
    h = np.sqrt(mu * p)
    v_orb = np.array([-(mu/h) * np.sin(nu), (mu/h) * (e + np.cos(nu)), 0.0])
    
    # 3. Matrices de rotación para transformar al marco inercial absoluto
    R3_Omega = np.array([
        [np.cos(Omega), -np.sin(Omega), 0],
        [np.sin(Omega),  np.cos(Omega), 0],
        [0,              0,             1]
    ])
    R1_i = np.array([
        [1, 0,         0],
        [0, np.cos(i), -np.sin(i)],
        [0, np.sin(i),  np.cos(i)]
    ])
    R3_omega = np.array([
        [np.cos(omega), -np.sin(omega), 0],
        [np.sin(omega),  np.cos(omega), 0],
        [0,              0,             1]
    ])
    
    # Matriz de transformación total
    Q = R3_Omega @ R1_i @ R3_omega
    
    r_cartesiano = Q @ r_orb
    v_cartesiano = Q @ v_orb
    
    return r_cartesiano, v_cartesiano


# =====================================================================
# SECCIÓN 7: MÓDULO DE EXPERIMENTOS Y CASOS DE SIMULACIÓN (ACTUALIZADO)
# =====================================================================

# Constantes Globales (Asegúrate de tener definidas tus constantes base aquí)
G = 6.67430e-11
G_const = G
M_T = 5.972e24
M_L = 7.342e22
R_T = 6371e3
D_TL = 384400e3

# =============================================================================
# FUNCIONES AUXILIARES Y DE INTEGRACIÓN
# =============================================================================
def graficar_esfera(ax, radio, centro, color):
    """
    Genera una esfera 3D en el gráfico (Axes3D) para reemplazar los puntos simples.
    """
    u = np.linspace(0, 2 * np.pi, 50)
    v = np.linspace(0, np.pi, 50)
    x = centro[0] + radio * np.outer(np.cos(u), np.sin(v))
    y = centro[1] + radio * np.outer(np.sin(u), np.sin(v))
    z = centro[2] + radio * np.outer(np.ones(np.size(u)), np.cos(v))
    
    # CORRECCIÓN: Se elimina edgecolors='none' y se usa linewidth=0
    ax.plot_surface(x, y, z, color=color, alpha=0.8, linewidth=0, antialiased=False)



# =============================================================================
# CONDICIONES INICIALES BASE (Marco Baricéntrico)
# =============================================================================
M_tot = M_T + M_L

# 1. Posiciones respecto al centro de masas
r_T = D_TL * (M_L / M_tot)
r_L = D_TL * (M_T / M_tot)

# 2. Velocidad relativa exacta para órbita circular de 2 cuerpos
v_rel = np.sqrt(G * M_tot / D_TL)

# 3. Velocidades respecto al centro de masas (Momentos opuestos)
v_T = v_rel * (M_L / M_tot)
v_L = v_rel * (M_T / M_tot)

# Aplicar las nuevas condiciones
pos_Tierra = np.array([-r_T, 0.0, 0.0])
vel_Tierra = np.array([0.0, -v_T, 0.0])
pos_Luna = np.array([r_L, 0.0, 0.0])
vel_Luna = np.array([0.0, v_L, 0.0])

def fijar_escala_3d_igual(ax):
    """
    Fuerza a que los 3 ejes de un gráfico 3D tengan exactamente 
    la misma escala matemática para evitar deformaciones geométricas.
    """
    ax.set_box_aspect([1, 1, 1]) # Caja visual cúbica
    
    # Obtener los límites actuales
    x_lim, y_lim, z_lim = ax.get_xlim3d(), ax.get_ylim3d(), ax.get_zlim3d()
    
    # Calcular los rangos y centros de cada eje
    x_r, x_m = x_lim[1] - x_lim[0], np.mean(x_lim)
    y_r, y_m = y_lim[1] - y_lim[0], np.mean(y_lim)
    z_r, z_m = z_lim[1] - z_lim[0], np.mean(z_lim)
    
    # Encontrar el rango máximo y definir el radio de la caja
    max_r = max(x_r, y_r, z_r) / 2.0
    
    # Aplicar los nuevos límites simétricos y equidistantes
    ax.set_xlim3d([x_m - max_r, x_m + max_r])
    ax.set_ylim3d([y_m - max_r, y_m + max_r])
    ax.set_zlim3d([z_m - max_r, z_m + max_r])

# =============================================================================
# TAREA 1: SIMULACIÓN DE 2 CUERPOS (Tierra, ISS)
# =============================================================================
def ejecutar_tarea_1():

    h = 15.0  

    print(f"\nIniciando Tarea 1: Simulación de 2 Cuerpos (h = {h}s)...")
    
    dias_simulacion = 28
    t_max = dias_simulacion * 24 * 3600
    t_eval = np.arange(0, t_max, h)
    N_pasos = len(t_eval)

    pos_T = pos_Tierra 
    vel_T = vel_Tierra

    altitud_iss = 400e3
    r_iss = R_T + altitud_iss
    v_iss = np.sqrt(G * M_T / r_iss)
    pos_ISS = pos_T + np.array([r_iss, 0.0, 0.0])
    vel_ISS = vel_T + np.array([0.0, v_iss, 0.0])

    posiciones = np.array([pos_T, pos_ISS])
    velocidades = np.array([vel_T, vel_ISS])
    masas = np.array([M_T, 419725]) 

    hist_pos = np.zeros((N_pasos, 2, 3))
    hist_pos[0] = posiciones

    print(f"Integrando {N_pasos} pasos para la Tarea 1...")
    for i in range(1, N_pasos):
        posiciones, velocidades = paso_rk4(masas, posiciones, velocidades, h, G)
        hist_pos[i] = posiciones

    hist_tierra = hist_pos[:, 0, :]
    hist_iss    = hist_pos[:, 1, :]

    print("Integración completada. Generando gráficas de la Tarea 1...")

    # Límite de pasos para graficar (Evitar bloque verde sólido: Solo graficamos ~3 órbitas)
    limite_plot = int((3 * 90 * 60) / h)

    # --- T1.1: Sistema Completo 3D ---
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    # Uso de esfera en lugar de línea para la Tierra
    graficar_esfera(ax, R_T, hist_tierra[0], 'blue')
    ax.plot(hist_iss[:limite_plot,0], hist_iss[:limite_plot,1], hist_iss[:limite_plot,2], 
            'g-', linewidth=0.8, label='ISS (LEO - 3 Órbitas)')
    ax.set_title('T1: Trayectoria 3D Sistema Tierra-ISS')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
    fijar_escala_3d_igual(ax)
    
    # Ajuste de límites para que la esfera sea visible proporcionalmente
    r_plot = r_iss * 1.2
    ax.set_xlim([-r_plot, r_plot])
    ax.set_ylim([-r_plot, r_plot])
    ax.set_zlim([-r_plot, r_plot])
    
    # Fake plot for legend
    ax.plot([], [], 'o', color='blue', label='Tierra (Esfera real)')
    ax.legend()
    plt.tight_layout()
    plt.savefig("T1_01_Sistema_3D.png", dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()

    # --- T1.2, T1.3, T1.4: Proyecciones 2D (Con límite de órbitas) ---
    # --- T1.2, T1.3, T1.4: Proyecciones 2D (Con límite de órbitas) ---
    planos = [('XY', 0, 1), ('XZ', 0, 2), ('YZ', 1, 2)]
    for idx, (nombre_plano, c1, c2) in enumerate(planos):
        plt.figure(figsize=(8, 8))
        
        # CORRECCIÓN: Leer el centro real de la Tierra en el plano actual
        centro_x = hist_tierra[0, c1]
        centro_y = hist_tierra[0, c2]
        circle = plt.Circle((centro_x, centro_y), R_T, color='blue', label='Tierra')
        
        plt.gca().add_patch(circle)
        plt.plot(hist_iss[:limite_plot,c1], hist_iss[:limite_plot,c2], 'g-', linewidth=0.8, label='ISS (LEO)')
        plt.title(f'T1: Proyección 2D Plano {nombre_plano}')
        plt.xlabel(f'Eje {nombre_plano[0]} (m)')
        plt.ylabel(f'Eje {nombre_plano[1]} (m)')
        plt.axis('equal')
        plt.legend()
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.tight_layout()
        plt.savefig(f"T1_0{2+idx}_Sistema_{nombre_plano}.png", dpi=600, bbox_inches='tight', facecolor='white')
        plt.close()

    # --- T1.5: Enfoque ISS 3D ---
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(hist_iss[:limite_plot,0], hist_iss[:limite_plot,1], hist_iss[:limite_plot,2], 
            'g-', linewidth=1.0, label='Órbita ISS')
    graficar_esfera(ax, R_T, hist_tierra[0], 'blue')
    ax.set_title('T1: Trayectoria 3D Detallada (ISS)')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
    ax.set_xlim([-r_plot, r_plot]); ax.set_ylim([-r_plot, r_plot]); ax.set_zlim([-r_plot, r_plot])
    ax.plot([], [], 'o', color='blue', label='Tierra')
    fijar_escala_3d_igual(ax)
    ax.legend()
    plt.savefig("T1_05_ISS_Enfoque_3D.png", dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()

    # --- T1.6: Variable en el Tiempo (Toda la simulación) ---
    tiempo_dias = t_eval / (24 * 3600)
    dist_iss = np.linalg.norm(hist_iss - hist_tierra, axis=1) / 1000.0  
    
    plt.figure(figsize=(10, 5))
    plt.plot(tiempo_dias, dist_iss, 'g-')
    plt.title(f'T1: Variación Altitud ISS (h = {h}s)')
    plt.xlabel('Tiempo (Días)')
    plt.ylabel('Distancia (km)')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("T1_06_Distancias_Temporales.png", dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print("Gráficas Tarea 1 exportadas.")

# =============================================================================
# TAREA 2: SIMULACIÓN DE 5 CUERPOS (Tierra, Luna, ISS, JWST, Sat L4)
# =============================================================================
def ejecutar_tarea_2():
    
    h = 15.0  

    print(f"\nIniciando Tarea 2: Simulación de 5 Cuerpos (h = {h}s)...")
    
    dias_simulacion = 28
    t_max = dias_simulacion * 24 * 3600
    t_eval = np.arange(0, t_max, h)
    N_pasos = len(t_eval)

    altitud_iss = 400e3
    r_iss = R_T + altitud_iss
    v_iss = np.sqrt(G * M_T / r_iss)
    pos_ISS = pos_Tierra + np.array([r_iss, 0.0, 0.0])
    vel_ISS = vel_Tierra + np.array([0.0, v_iss, 0.0])

    q = M_L / M_T
    L2_dist_ratio = (q/3)**(1/3) + (1/3)*(q/3)**(2/3) - (1/9)*(q/3)
    r_h_exacto = D_TL * L2_dist_ratio
    pos_JWST = pos_Luna + np.array([r_h_exacto, 0.0, 0.0])
    omega_sistema = np.sqrt(G * (M_T + M_L) / D_TL**3)
    vel_JWST = np.array([0.0, omega_sistema * (pos_JWST[0] - pos_Tierra[0]), 0.0])

    theta_L4 = np.pi / 3.0  
    R_60 = np.array([
        [np.cos(theta_L4), -np.sin(theta_L4), 0.0],
        [np.sin(theta_L4),  np.cos(theta_L4), 0.0],
        [0.0,               0.0,              1.0]
    ])
    r_EM = pos_Luna - pos_Tierra
    v_EM = vel_Luna - vel_Tierra
    pos_L4_sat = pos_Tierra + R_60 @ r_EM
    vel_L4_sat = vel_Tierra + R_60 @ v_EM

    posiciones = np.array([pos_Tierra, pos_Luna, pos_ISS, pos_JWST, pos_L4_sat])
    velocidades = np.array([vel_Tierra, vel_Luna, vel_ISS, vel_JWST, vel_L4_sat])
    masas = np.array([M_T, M_L, 419725, 6161, 1000]) 

    hist_pos = np.zeros((N_pasos, 5, 3))
    hist_pos[0] = posiciones

    print(f"Integrando {N_pasos} pasos para Tarea 2...")
    for i in range(1, N_pasos):
        posiciones, velocidades = paso_rk4(masas, posiciones, velocidades, h, G)
        hist_pos[i] = posiciones

    hist_tierra = hist_pos[:, 0, :]
    hist_luna   = hist_pos[:, 1, :]
    hist_iss    = hist_pos[:, 2, :]
    hist_jwst   = hist_pos[:, 3, :]
    hist_l4     = hist_pos[:, 4, :]

    print("Generando múltiples perspectivas para Tarea 2...")

    # Factores de escala para visibilidad de esferas (10x para que destaquen a nivel orbital)
    escala_tierra = R_T * 5
    escala_luna = 1737.4e3 * 5

    # --- T2.1: 3D Sistema Completo ---
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    limite_espacial = D_TL * 1.2 # Un 20% más allá de la órbita de la Luna
    
    ax.set_xlim3d([-limite_espacial, limite_espacial])
    ax.set_ylim3d([-limite_espacial, limite_espacial])
    ax.set_zlim3d([-limite_espacial, limite_espacial])
    ax.set_box_aspect([1, 1, 1])    
    ax.set_box_aspect([1, 1, 1]) 
        
    graficar_esfera(ax, escala_tierra, hist_tierra[0], 'blue')
    graficar_esfera(ax, escala_luna, hist_luna[0], 'gray')
    
    ax.plot(hist_luna[:,0], hist_luna[:,1], hist_luna[:,2], color='gray', linestyle='-', label='Órbita Luna')
    ax.plot(hist_jwst[:,0], hist_jwst[:,1], hist_jwst[:,2], 'r-', label='JWST (L2)')
    # Diferenciación visual del Satélite L4 con línea punteada púrpura
    ax.plot(hist_l4[:,0], hist_l4[:,1], hist_l4[:,2], color='purple', linestyle='--', linewidth=1.5, label='Satélite (L4)')
    
    ax.set_title('T2: Trayectoria 3D de 5 Cuerpos')
    ax.plot([], [], 'o', color='blue', label='Tierra (Escala x5)')
    ax.plot([], [], 'o', color='gray', label='Luna (Escala x5)')
    ax.legend()
    plt.tight_layout()
    plt.savefig("T2_01_Sistema_3D.png", dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()

    # --- T2.2, T2.3, T2.4: Planos 2D ---
    planos = [('XY', 0, 1), ('XZ', 0, 2), ('YZ', 1, 2)]
    for idx, (nombre, c1, c2) in enumerate(planos):
        plt.figure(figsize=(9, 9))
        plt.plot(hist_tierra[:,c1], hist_tierra[:,c2], 'b-', label='Tierra')
        plt.plot(hist_luna[:,c1], hist_luna[:,c2], color='gray', linestyle='-', label='Luna')
        plt.plot(hist_jwst[:,c1], hist_jwst[:,c2], 'r-', label='JWST (L2)')
        # Diferenciación visual en planos 2D
        plt.plot(hist_l4[:,c1], hist_l4[:,c2], color='purple', linestyle='--', linewidth=1.5, label='Sat L4')
        plt.title(f'T2: Dinámica Lagrange en Plano {nombre}')
        plt.xlabel(f'Eje {nombre[0]} (m)')
        plt.ylabel(f'Eje {nombre[1]} (m)')
        plt.axis('equal')
        plt.legend()
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.tight_layout()
        plt.savefig(f"T2_0{2+idx}_Lagrange_{nombre}.png", dpi=600, bbox_inches='tight', facecolor='white')
        plt.close()

    # --- T2.5: Variables Temporales ---
    tiempo_dias = t_eval / (24 * 3600)
    dist_jwst = np.linalg.norm(hist_jwst - hist_tierra, axis=1) / 1000.0
    dist_l4 = np.linalg.norm(hist_l4 - hist_tierra, axis=1) / 1000.0

    plt.figure(figsize=(10, 6))
    plt.plot(tiempo_dias, dist_jwst, 'r-', label="JWST (Inestabilidad L2)")
    plt.plot(tiempo_dias, dist_l4, 'm-', label="Satélite (Estabilidad L4)")
    plt.title('T2: Evolución de Distancias Relativas (Tiempo)')
    plt.xlabel('Tiempo (Días)')
    plt.ylabel('Distancia a la Tierra (km)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("T2_05_Evolucion_Distancias.png", dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()
    print("Gráficas Tarea 2 exportadas.")

# =============================================================================
# TAREA 3: EXPANSIÓN N-CUERPOS (Sistema Solar Interior + Lunas)
# =============================================================================
def ejecutar_tarea_3():
    print("\n--- Ejecutando Tarea 3: Sistema Solar Interior (N-Cuerpos 3D) ---")
    
    nombres = ["Sol", "Mercurio", "Venus", "Tierra", "Luna", "Marte", "Fobos", "Deimos"]
    masas = np.array([1.989e30, 3.301e23, 4.867e24, 5.972e24, 7.342e22, 6.417e23, 1.06e16, 1.48e15])
    N = len(masas)
    
    mu_sol = G_const * masas[0]
    # Corrección: Forzar la conversión de los grados a radianes explícitamente para evitar la órbita perpendicular
    r_mercurio, v_mercurio = kepler_a_cartesiano(
        a=57.9e9, e=0.2056, 
        i_deg=7.0, 
        Omega_deg=48.33, 
        omega_deg=29.124, 
        M=0.0, mu=mu_sol
    )
    
    pos_ini = np.zeros((N, 3))
    vel_ini = np.zeros((N, 3))
    
    pos_ini[1] = r_mercurio
    vel_ini[1] = v_mercurio
    pos_ini[2] = [108.2e9, 0, 0]; vel_ini[2] = [0, 35020, 0]
    pos_ini[3] = [149.6e9, 0, 0]; vel_ini[3] = [0, 29780, 0]
    pos_ini[4] = pos_ini[3] + [3.84e8, 0, 0]; vel_ini[4] = vel_ini[3] + [0, 1022, 0]
    pos_ini[5] = [227.9e9, 0, 0]; vel_ini[5] = [0, 24070, 0]
    pos_ini[6] = pos_ini[5] + [9.37e6, 0, 0]; vel_ini[6] = vel_ini[5] + [0, 2138, 0]
    pos_ini[7] = pos_ini[5] + [23.4e6, 0, 0]; vel_ini[7] = vel_ini[5] + [0, 1351, 0]
    
    t_final = 86400 * 730 
    h_paso = 3600.0       
    
    log_t, log_pos, log_vel, E, L = simular_y_registrar(
        masas, pos_ini, vel_ini, t_final, h_paso, G_const, integrador='simplectico'
    )
    
    print("Mapeo completado. Exportando perspectivas completas y variables temporales...")

    # --- T3.1: 3D Sistema Solar Interior ---
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    colores = ['orange', 'gray', 'goldenrod', 'blue', 'silver', 'red', 'darkred', 'salmon']
    for i in range(N):
        ax.plot(log_pos[:, i, 0], log_pos[:, i, 1], log_pos[:, i, 2], color=colores[i], label=nombres[i])
    
    # Ajustar relación de aspecto 3D (Workaround para matplotlib) para que el plano eclíptico se vea proporcionado
    max_range = np.array([
        log_pos[:, :, 0].max() - log_pos[:, :, 0].min(),
        log_pos[:, :, 1].max() - log_pos[:, :, 1].min(),
        log_pos[:, :, 2].max() - log_pos[:, :, 2].min()
    ]).max() / 2.0
    
    mid_x = (log_pos[:, :, 0].max() + log_pos[:, :, 0].min()) * 0.5
    mid_y = (log_pos[:, :, 1].max() + log_pos[:, :, 1].min()) * 0.5
    mid_z = (log_pos[:, :, 2].max() + log_pos[:, :, 2].min()) * 0.5
    
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    ax.set_title('T3: Sistema Solar Interior (3D)')
    ax.legend(loc='upper right', bbox_to_anchor=(1.15, 1))
    plt.savefig("T3_01_SistemaSolar_3D.png", dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()

    # --- T3.2, T3.3, T3.4: Planos 2D ---
    planos = [('XY', 0, 1), ('XZ', 0, 2), ('YZ', 1, 2)]
    for idx, (nombre, c1, c2) in enumerate(planos):
        plt.figure(figsize=(10, 10))
        for i in range(N):
            plt.plot(log_pos[:, i, c1], log_pos[:, i, c2], color=colores[i], label=nombres[i])
        plt.title(f'T3: Sistema Solar Interior en Plano {nombre}')
        plt.xlabel(f'Eje {nombre[0]} (m)'); plt.ylabel(f'Eje {nombre[1]} (m)')
        plt.axis('equal')
        plt.legend()
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.tight_layout()
        plt.savefig(f"T3_0{2+idx}_SistemaSolar_{nombre}.png", dpi=600, bbox_inches='tight', facecolor='white')
        plt.close()

    # --- T3.5: Evolución Temporal Z ---
    plt.figure(figsize=(10, 5))
    plt.plot(log_t / 86400, log_pos[:, 1, 2] / 1e9, label="Mercurio (7°)")
    plt.plot(log_t / 86400, log_pos[:, 3, 2] / 1e9, label="Tierra (0°)")
    plt.title("T3: Oscilación Z en el Tiempo (Órbitas No Coplanares)")
    plt.xlabel("Tiempo (Días)"); plt.ylabel("Elevación Z (10^6 km)")
    plt.legend()
    plt.grid(True)
    plt.savefig("T3_05_Inclinacion_Z.png", dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()

    # --- T3.6: Conservación de Energía y Momento ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    ax1.plot(log_t / 86400, E, color='red', label='Energía Total (E)')
    ax1.set_ylabel("Energía (J)")
    ax1.set_title("T3: Conservación de Variables Físicas en el Tiempo")
    ax1.grid(True); ax1.legend()
    
    magnitud_L = np.linalg.norm(L, axis=1)
    ax2.plot(log_t / 86400, magnitud_L, color='blue', label='Magnitud ||L||')
    ax2.plot(log_t / 86400, L[:, 0], '--', label='Lx', alpha=0.5)
    ax2.plot(log_t / 86400, L[:, 1], '--', label='Ly', alpha=0.5)
    ax2.plot(log_t / 86400, L[:, 2], '--', label='Lz', alpha=0.5)
    ax2.set_xlabel("Tiempo (Días)"); ax2.set_ylabel("Momento Angular (L)")
    ax2.grid(True); ax2.legend()
    plt.tight_layout()
    plt.savefig("T3_06_Conservacion_E_L.png", dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print("Gráficas Tarea 3 exportadas exitosamente.")

# =============================================================================
# EJECUCIÓN PRINCIPAL
# =============================================================================
if __name__ == "__main__":
    ejecutar_tarea_1()
    ejecutar_tarea_2()
    ejecutar_tarea_3()


Iniciando Tarea 1: Simulación de 2 Cuerpos (h = 15.0s)...
Integrando 161280 pasos para la Tarea 1...
Integración completada. Generando gráficas de la Tarea 1...
Gráficas Tarea 1 exportadas.

Iniciando Tarea 2: Simulación de 5 Cuerpos (h = 15.0s)...
Integrando 161280 pasos para Tarea 2...
Generando múltiples perspectivas para Tarea 2...
Gráficas Tarea 2 exportadas.

--- Ejecutando Tarea 3: Sistema Solar Interior (N-Cuerpos 3D) ---
Mapeo completado. Exportando perspectivas completas y variables temporales...
Gráficas Tarea 3 exportadas exitosamente.
